# 09 – Khử mẹo trùng từ cho PhoBERT (W7-01)

**Người chạy: Quang.** Tổng khoảng **2 giờ** GPU T4 (có thể chia làm 2 lần, phần nào xong sẽ tự bỏ qua).

**Ý tưởng:** PhoBERT học trên XNLI (S1) chỉ đạt khoảng 28% trên ViANLI vì học mẹo "hai câu trùng nhiều từ → suy ra được".
M4P huấn luyện PhoBERT kèm một **mô hình mẹo** (Logistic Regression chỉ dùng 5 đặc trưng bề mặt của M1: trùng từ, phủ định,
độ dài, số). Hai mô hình được kết hợp khi tính loss (product of experts), nên PhoBERT bị "trừ" phần mà mẹo đã đoán được và phải
học phần còn lại. Khi dự đoán chỉ dùng PhoBERT. Code: `src/debias.py`, `src/transformer.py` (mô hình `M4P`).

Notebook chạy:
1. M4P kịch bản S1, 3 seed (42, 123, 2024), khoảng 25 phút mỗi seed;
2. PhoBERT thường (M4) S1 thêm seed 123, 2024 để so công bằng với M4P (3 seed mỗi bên), khoảng 25 phút mỗi seed.

**Bắt buộc dùng GPU:** Runtime → Change runtime type → **T4 GPU** → Save.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results', 'checkpoints']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

## Kiểm tra GPU và dữ liệu đã tách từ (PhoBERT cần)

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU, bật T4 GPU rồi chạy lại từ ô đầu')
need = [f'data/processed/{f}_seg.jsonl' for f in ['xnli_train_100k', 'xnli_dev', 'xnli_test', 'vianli_test', 'vinli_test']]
missing = [f for f in need if not os.path.exists(f)]
print('Thiếu file tách từ:', missing if missing else 'không, đủ')

**Đạt khi:** GPU `Tesla T4`, không thiếu file. Nếu thiếu file tách từ thì chạy ô dưới (cần Java, Colab có sẵn, khoảng 15–20 phút), đủ rồi thì bỏ qua.

In [ ]:
!python scripts/segment_data.py

## Bước 1: chạy thử M4P (khoảng 3 phút)

In [ ]:
!python scripts/run_transformer.py --model M4P --scenario S1 --smoke

**Đạt khi:** không có `Traceback`, cuối ô có 3 dòng `[M4P_S1_..._seed42] accuracy=...` (chạy thử, lưu ở `results/smoke/`, điểm thấp là bình thường).

## Bước 2: M4P S1, 3 seed (khoảng 75 phút)

In [ ]:
!python scripts/run_transformer.py --model M4P --scenario S1 --config A --seeds 42 123 2024

## Bước 3: PhoBERT thường S1, thêm seed 123 và 2024 (khoảng 50 phút)
Seed 42 đã có từ trước nên chỉ chạy 2 seed mới.

In [ ]:
!python scripts/run_transformer.py --model M4 --scenario S1 --config A --seeds 123 2024

## Bước 4: kiểm tra

In [ ]:
!python scripts/check_w7.py --task 01

**Đạt khi:** dòng cuối ghi `Đủ 15 file kết quả. Xong W7-01.`

## Lưu và nộp
Kết quả tự nằm trên Drive `NLI/results/`. Sheet: W7-01 → **Đang review**, nhắn Quân để đồng bộ và phân tích.